<a href="https://colab.research.google.com/github/benlebosss/Adaltas-ECE-2026/blob/main/lab_2_Taxi_sparksql_and_dataframes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [8]:
!pip install pyspark requests

In [9]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [10]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [11]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [12]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [13]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Clé unique et durée en minutes
df_trips = df_trips.withColumn("trip_id", F.monotonically_increasing_id()) \
                   .withColumn("duration_minutes",
                               (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0)

# Nettoyage de base : restreindre à janvier 2019 et durées positives
clean_trips = df_trips.filter(
    (F.col("duration_minutes") > 0) &
    (F.col("tpep_pickup_datetime").between("2019-01-01", "2019-01-31 23:59:59"))
)

clean_trips.select("trip_id", "passenger_count", "trip_distance", "duration_minutes").show(5)

+-------+---------------+-------------+------------------+
|trip_id|passenger_count|trip_distance|  duration_minutes|
+-------+---------------+-------------+------------------+
|      0|            1.0|          1.5| 6.666666666666667|
|      1|            1.0|          2.6|              19.2|
|      7|            1.0|          1.3|              7.15|
|      8|            1.0|          3.7|13.633333333333333|
|      9|            2.0|          2.1|              12.0|
+-------+---------------+-------------+------------------+
only showing top 5 rows


In [14]:
# Trajet(s) avec le maximum de passagers
max_passengers = clean_trips.select(F.max("passenger_count")).first()[0]
print(f"Nombre maximum de passagers : {max_passengers}")
clean_trips.filter(F.col("passenger_count") == max_passengers) \
           .select("trip_id", "passenger_count", "trip_distance", "total_amount").show(5)

# Moyenne des passagers
avg_passengers = clean_trips.select(F.avg("passenger_count")).first()[0]
print(f"Nombre moyen de passagers : {avg_passengers:.2f}")

Nombre maximum de passagers : 9.0
+-------+---------------+-------------+------------+
|trip_id|passenger_count|trip_distance|total_amount|
+-------+---------------+-------------+------------+
| 949956|            9.0|          0.0|        12.6|
|1296287|            9.0|          0.0|         9.3|
|2012098|            9.0|          0.0|        11.3|
|2883995|            9.0|          0.0|       12.25|
|4534707|            9.0|          0.0|      110.76|
+-------+---------------+-------------+------------+
only showing top 5 rows
Nombre moyen de passagers : 1.57


In [15]:
print("--- Par distance ---")
print("Plus court :")
clean_trips.orderBy(F.asc("trip_distance")).select("trip_id", "trip_distance", "duration_minutes").show(1)

print("Plus long :")
clean_trips.orderBy(F.desc("trip_distance")).select("trip_id", "trip_distance", "duration_minutes").show(1)

print("--- Par durée ---")
print("Plus court :")
clean_trips.orderBy(F.asc("duration_minutes")).select("trip_id", "trip_distance", "duration_minutes").show(1)

print("Plus long :")
clean_trips.orderBy(F.desc("duration_minutes")).select("trip_id", "trip_distance", "duration_minutes").show(1)

--- Par distance ---
Plus court :
+-------+-------------+------------------+
|trip_id|trip_distance|  duration_minutes|
+-------+-------------+------------------+
|    845|          0.0|2.1166666666666667|
+-------+-------------+------------------+
only showing top 1 row
Plus long :
+-------+-------------+-----------------+
|trip_id|trip_distance| duration_minutes|
+-------+-------------+-----------------+
|6074091|        831.8|9.483333333333333|
+-------+-------------+-----------------+
only showing top 1 row
--- Par durée ---
Plus court :
+-------+-------------+--------------------+
|trip_id|trip_distance|    duration_minutes|
+-------+-------------+--------------------+
|  12355|          0.0|0.016666666666666666|
+-------+-------------+--------------------+
only showing top 1 row
Plus long :
+-------+-------------+-----------------+
|trip_id|trip_distance| duration_minutes|
+-------+-------------+-----------------+
|  68267|          1.2|43648.01666666667|
+-------+-------------+-

In [16]:
df_daily = clean_trips.withColumn("pickup_date", F.to_date("tpep_pickup_datetime")) \
                      .groupBy("pickup_date") \
                      .count() \
                      .orderBy(F.desc("count"))

print("Jour le plus chargé :")
df_daily.show(1)

print("Jour le moins chargé :")
df_daily.orderBy(F.asc("count")).show(1)

Jour le plus chargé :
+-----------+------+
|pickup_date| count|
+-----------+------+
| 2019-01-25|292235|
+-----------+------+
only showing top 1 row
Jour le moins chargé :
+-----------+------+
|pickup_date| count|
+-----------+------+
| 2019-01-01|189276|
+-----------+------+
only showing top 1 row


In [17]:
df_time_bucket = clean_trips.withColumn("hour", F.hour("tpep_pickup_datetime")) \
    .withColumn("time_of_day",
        F.when(F.col("hour").between(6, 11), "Morning")
         .when(F.col("hour").between(12, 17), "Afternoon")
         .when(F.col("hour").between(18, 23), "Evening")
         .otherwise("Late Night")
    )

print("Fréquentation par moment de la journée :")
df_time_bucket.groupBy("time_of_day").count().orderBy(F.desc("count")).show()

Fréquentation par moment de la journée :
+-----------+-------+
|time_of_day|  count|
+-----------+-------+
|  Afternoon|2577951|
|    Evening|2472476|
|    Morning|1958346|
| Late Night| 680753|
+-----------+-------+



In [18]:
df_dow = clean_trips.withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "E")) \
                    .withColumn("date", F.to_date("tpep_pickup_datetime")) \
                    .groupBy("day_of_week", "date") \
                    .count() \
                    .groupBy("day_of_week") \
                    .agg(F.round(F.avg("count"), 0).alias("avg_trips_per_day")) \
                    .orderBy(F.desc("avg_trips_per_day"))

df_dow.show()

+-----------+-----------------+
|day_of_week|avg_trips_per_day|
+-----------+-----------------+
|        Fri|         271543.0|
|        Thu|         271170.0|
|        Wed|         252828.0|
|        Sat|         252301.0|
|        Tue|         241614.0|
|        Mon|         226732.0|
|        Sun|         214791.0|
+-----------+-----------------+



In [19]:
# Corrélations
clean_trips.select(
    F.corr("trip_distance", "tip_amount").alias("corr_distance_tip"),
    F.corr("passenger_count", "tip_amount").alias("corr_passengers_tip")
).show()

# Moyenne du tip par tranche de 5 miles
clean_trips.withColumn("dist_bin", F.floor(F.col("trip_distance") / 5) * 5) \
           .groupBy("dist_bin") \
           .agg(F.round(F.avg("tip_amount"), 2).alias("avg_tip"), F.count("*").alias("nb_trips")) \
           .filter(F.col("dist_bin") <= 30) \
           .orderBy("dist_bin").show()

+------------------+--------------------+
| corr_distance_tip| corr_passengers_tip|
+------------------+--------------------+
|0.5269469937968689|9.458538888937504E-4|
+------------------+--------------------+

+--------+-------+--------+
|dist_bin|avg_tip|nb_trips|
+--------+-------+--------+
|       0|   1.39| 6662542|
|       5|   3.47|  585763|
|      10|   4.95|  210621|
|      15|   7.26|  180848|
|      20|   7.36|   39640|
|      25|   8.12|    6962|
|      30|  10.79|    1538|
+--------+-------+--------+



In [20]:
clean_trips.orderBy(F.desc("extra")) \
           .select("trip_id", "extra", "fare_amount", "total_amount", "tpep_pickup_datetime") \
           .show(5)

+-------+-----+-----------+------------+--------------------+
|trip_id|extra|fare_amount|total_amount|tpep_pickup_datetime|
+-------+-----+-----------+------------+--------------------+
| 311052| 18.5|       52.0|        88.3| 2019-01-02 16:33:28|
|2455086| 18.5|       49.0|       96.36| 2019-01-11 16:08:48|
| 134549| 18.5|       39.5|        70.8| 2019-01-01 16:09:32|
| 543203| 18.5|       61.0|        92.3| 2019-01-03 18:32:36|
| 548308| 18.5|       47.5|       94.56| 2019-01-03 18:19:33|
+-------+-----+-----------+------------+--------------------+
only showing top 5 rows


In [21]:
# Identification de valeurs suspectes
outliers = clean_trips.filter(
    (F.col("fare_amount") < 0) |
    ((F.col("trip_distance") == 0) & (F.col("fare_amount") > 50)) |
    (F.col("duration_minutes") > 1440) |
    (F.col("passenger_count") == 0)
)

print(f"Nombre d'enregistrements aberrants : {outliers.count()}")
outliers.select("trip_id", "fare_amount", "trip_distance", "duration_minutes", "passenger_count").show(5)

Nombre d'enregistrements aberrants : 136595
+-------+-----------+-------------+------------------+---------------+
|trip_id|fare_amount|trip_distance|  duration_minutes|passenger_count|
+-------+-----------+-------------+------------------+---------------+
|    156|        2.5|          5.3|              0.95|            0.0|
|    228|       52.0|         18.0|30.966666666666665|            0.0|
|    229|       29.5|          8.9|31.316666666666666|            0.0|
|    298|        8.0|          1.0|10.566666666666666|            0.0|
|    663|       -2.5|          0.1|              0.65|            2.0|
+-------+-----------+-------------+------------------+---------------+
only showing top 5 rows


### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [22]:
lookup_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
resp_lookup = requests.get(lookup_url)
if resp_lookup.status_code == 200:
    with open("taxi_zone_lookup.csv", "wb") as f:
        f.write(resp_lookup.content)

df_zones = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)
df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [23]:
# Pickups
df_pickup = clean_trips.join(df_zones, clean_trips.PULocationID == df_zones.LocationID, "inner") \
                       .withColumnRenamed("Borough", "Pickup_Borough") \
                       .withColumnRenamed("Zone", "Pickup_Zone")

print("Top Pickups par Borough :")
df_pickup.groupBy("Pickup_Borough").count().orderBy(F.desc("count")).show()

# Dropoffs
df_dropoff = clean_trips.join(df_zones, clean_trips.DOLocationID == df_zones.LocationID, "inner") \
                        .withColumnRenamed("Borough", "Dropoff_Borough")

print("Top Dropoffs par Borough :")
df_dropoff.groupBy("Dropoff_Borough").count().orderBy(F.desc("count")).show()

Top Pickups par Borough :
+--------------+-------+
|Pickup_Borough|  count|
+--------------+-------+
|     Manhattan|6946409|
|        Queens| 470428|
|       Unknown| 158238|
|      Brooklyn|  91755|
|         Bronx|  18022|
|           N/A|   3871|
|           EWR|    443|
| Staten Island|    360|
+--------------+-------+

Top Dropoffs par Borough :
+---------------+-------+
|Dropoff_Borough|  count|
+---------------+-------+
|      Manhattan|6816488|
|         Queens| 340724|
|       Brooklyn| 300991|
|        Unknown| 143289|
|          Bronx|  58046|
|            N/A|  16893|
|            EWR|  10911|
|  Staten Island|   2184|
+---------------+-------+



In [24]:
# Heure la plus chargée par Borough
df_pickup_time = df_pickup.withColumn("hour", F.hour("tpep_pickup_datetime"))
w_hour = Window.partitionBy("Pickup_Borough").orderBy(F.desc("count"))

print("Heure de pointe par Borough :")
df_pickup_time.groupBy("Pickup_Borough", "hour") \
              .count() \
              .withColumn("rank", F.row_number().over(w_hour)) \
              .filter(F.col("rank") == 1) \
              .select("Pickup_Borough", F.col("hour").alias("peak_hour"), F.col("count").alias("trips")) \
              .show()

# Jour de la semaine le plus chargé par Borough
df_pickup_dow = df_pickup.withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE"))
w_dow = Window.partitionBy("Pickup_Borough").orderBy(F.desc("count"))

print("Jour le plus fréquenté par Borough :")
df_pickup_dow.groupBy("Pickup_Borough", "day_of_week") \
             .count() \
             .withColumn("rank", F.row_number().over(w_dow)) \
             .filter(F.col("rank") == 1) \
             .select("Pickup_Borough", F.col("day_of_week").alias("busiest_day"), "count") \
             .show()

Heure de pointe par Borough :
+--------------+---------+------+
|Pickup_Borough|peak_hour| trips|
+--------------+---------+------+
|         Bronx|        7|  1797|
|      Brooklyn|        8|  6926|
|           EWR|       15|    54|
|     Manhattan|       18|471241|
|           N/A|       19|   214|
|        Queens|       16| 29827|
| Staten Island|        8|    36|
|       Unknown|       18| 10642|
+--------------+---------+------+

Jour le plus fréquenté par Borough :
+--------------+-----------+-------+
|Pickup_Borough|busiest_day|  count|
+--------------+-----------+-------+
|         Bronx|   Thursday|   3115|
|      Brooklyn|    Tuesday|  15767|
|           EWR|  Wednesday|     83|
|     Manhattan|   Thursday|1228786|
|           N/A|    Tuesday|    700|
|        Queens|   Thursday|  78865|
| Staten Island|     Friday|     64|
|       Unknown|   Thursday|  28649|
+--------------+-----------+-------+



In [25]:
df_pickup.groupBy("Pickup_Borough").agg(
    F.round(F.avg("trip_distance"), 2).alias("avg_distance_miles"),
    F.round(F.avg("fare_amount"), 2).alias("avg_fare_usd")
).orderBy(F.desc("avg_distance_miles")).show()

+--------------+------------------+------------+
|Pickup_Borough|avg_distance_miles|avg_fare_usd|
+--------------+------------------+------------+
| Staten Island|             12.54|       45.41|
|        Queens|              11.3|       35.15|
|         Bronx|              7.25|       26.26|
|      Brooklyn|              4.79|       18.64|
|           N/A|              3.21|       59.72|
|           EWR|              2.66|       76.38|
|       Unknown|              2.44|       12.43|
|     Manhattan|              2.23|       10.74|
+--------------+------------------+------------+



In [26]:
print("Courses aux montants les plus élevés :")
df_pickup.filter(F.col("fare_amount") > 0) \
         .orderBy(F.desc("fare_amount")) \
         .select("trip_id", "fare_amount", "Pickup_Borough", "Pickup_Zone") \
         .show(3)

print("Courses aux montants les plus bas :")
df_pickup.filter(F.col("fare_amount") > 0) \
         .orderBy(F.asc("fare_amount")) \
         .select("trip_id", "fare_amount", "Pickup_Borough", "Pickup_Zone") \
         .show(3)

Courses aux montants les plus élevés :
+-------+-----------+--------------+--------------------+
|trip_id|fare_amount|Pickup_Borough|         Pickup_Zone|
+-------+-----------+--------------+--------------------+
|2499655|  623259.86|     Manhattan|Upper East Side S...|
|2841024|     8000.3|     Manhattan|     Lenox Hill West|
| 478819|    6666.65|       Unknown|                 N/A|
+-------+-----------+--------------+--------------------+
only showing top 3 rows
Courses aux montants les plus bas :
+-------+-----------+--------------+--------------+
|trip_id|fare_amount|Pickup_Borough|   Pickup_Zone|
+-------+-----------+--------------+--------------+
|   2038|       0.01|           N/A|Outside of NYC|
|  17371|       0.01|     Manhattan|  East Chelsea|
|  10267|       0.01|     Manhattan|  East Chelsea|
+-------+-----------+--------------+--------------+
only showing top 3 rows


In [27]:
url_2025 = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet'
resp_2025 = requests.get(url_2025)

if resp_2025.status_code == 200:
    with open("yellow_tripdata_2025-01.parquet", "wb") as f:
        f.write(resp_2025.content)
    df_2025 = spark.read.parquet("yellow_tripdata_2025-01.parquet")

    m_2019 = clean_trips.select(
        F.lit("2019-01").alias("period"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip")
    )

    m_2025 = df_2025.filter(F.col("fare_amount") > 0).select(
        F.lit("2025-01").alias("period"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip")
    )

    m_2019.union(m_2025).show()

+-------+------------+--------+-------+
| period|avg_distance|avg_fare|avg_tip|
+-------+------------+--------+-------+
|2019-01|        2.83|   12.43|   1.82|
|2025-01|        5.39|   18.33|   3.09|
+-------+------------+--------+-------+



### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [28]:
# Création des vues SQL temporaires
clean_trips.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

# Question 1 en SQL : Trajet avec le maximum de passagers
print("--- Question 1 : Max passagers (SQL) ---")
spark.sql("""
    SELECT trip_id, passenger_count, trip_distance, total_amount
    FROM trips
    WHERE passenger_count = (SELECT MAX(passenger_count) FROM trips)
""").show(5)

# Question 2 en SQL : Jours les plus chargés
print("--- Question 2 : Jours les plus chargés (SQL) ---")
spark.sql("""
    SELECT
        DATE(tpep_pickup_datetime) AS pickup_date,
        COUNT(*) AS total_trips
    FROM trips
    GROUP BY DATE(tpep_pickup_datetime)
    ORDER BY total_trips DESC
    LIMIT 3
""").show()

# Question 3 en SQL (avec JOIN requis) : Distance et tarif moyens par Borough
print("--- Question 3 : Distance et tarif par Borough avec JOIN (SQL) ---")
spark.sql("""
    SELECT
        z.Borough AS pickup_borough,
        COUNT(t.trip_id) AS total_trips,
        ROUND(AVG(t.trip_distance), 2) AS avg_distance,
        ROUND(AVG(t.fare_amount), 2) AS avg_fare
    FROM trips t
    JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY total_trips DESC
""").show()

--- Question 1 : Max passagers (SQL) ---
+-------+---------------+-------------+------------+
|trip_id|passenger_count|trip_distance|total_amount|
+-------+---------------+-------------+------------+
| 949956|            9.0|          0.0|        12.6|
|1296287|            9.0|          0.0|         9.3|
|2012098|            9.0|          0.0|        11.3|
|2883995|            9.0|          0.0|       12.25|
|4534707|            9.0|          0.0|      110.76|
+-------+---------------+-------------+------------+
only showing top 5 rows
--- Question 2 : Jours les plus chargés (SQL) ---
+-----------+-----------+
|pickup_date|total_trips|
+-----------+-----------+
| 2019-01-25|     292235|
| 2019-01-11|     291440|
| 2019-01-17|     284367|
+-----------+-----------+

--- Question 3 : Distance et tarif par Borough avec JOIN (SQL) ---
+--------------+-----------+------------+--------+
|pickup_borough|total_trips|avg_distance|avg_fare|
+--------------+-----------+------------+--------+
|    

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing